IMPORTING LIBRARIES

In [ ]:
import pandas as pd
import pyodbc

In [ ]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=MSI\SQLEXPRESS;"
    "DATABASE=Ecommerce_Analysis;"
    "Trusted_Connection=yes;"
    )

In [ ]:
orders = pd.read_sql('SELECT * FROM orders', conn)
customers = pd.read_sql('SELECT * FROM customers', conn)
order_items = pd.read_sql('SELECT * FROM order_items', conn)
payments = pd.read_sql('SELECT * FROM payments', conn)
reviews = pd.read_sql('SELECT * FROM reviews', conn)
products = pd.read_sql('SELECT * FROM products', conn)

CHECK STRUCTURE OF DATASET

In [ ]:
orders.info()

In [ ]:
orders.head()

DETECT MISSING VALUE

In [ ]:
orders.isnull().sum()

HANDLE MISSING VALUES

In [ ]:
orders['delivered_flag'] = orders['order_delivered_customer_date'].notnull()

REMOVE DUPLICATES

In [ ]:
orders.duplicated().sum()

In [ ]:
# if duplicates exist drop 
orders = orders.drop_duplicates()

CONVERT DATE COLUMNS

In [ ]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

CREATE ANALYTICAL COLUMNS

In [ ]:
# ORDERS YEAR
orders['order_year'] = orders['order_purchase_timestamp'].dt.year

In [ ]:
# ORDERS MONTH
orders['order_month'] = orders['order_purchase_timestamp'].dt.month

In [ ]:
# ORDERS DAY
orders['order_day'] = orders['order_purchase_timestamp'].dt.day

CALCULATE DELIVERY TIME

In [ ]:
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days

In [ ]:
orders.head(100)

MERGE TABLES (CREATE FINAL DATASET)

In [ ]:
ecommerce_sales = orders.merge(customers, on='customer_id')
ecommerce_sales = ecommerce_sales.merge(order_items, on='order_id')
ecommerce_sales = ecommerce_sales.merge(products, on='product_id')
ecommerce_sales = ecommerce_sales.merge(payments, on='order_id')

SAVE CLEAN DATASET

In [ ]:
ecommerce_sales.to_csv('cleaned_ecommerce_dataset.csv', index=False)